In [1]:
import numpy as np
from environments.simple_trading_env import SimpleTradingEnv
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3 import PPO
import pandas as pd
import torch
import matplotlib.pyplot as plt

# === CONFIGURATION ===
DATA_SYMBOL = 'BTCUSDT'
DATA_TIMEFRAME = '5m'
DATA_PATH = f'data/binance-{DATA_SYMBOL}-{DATA_TIMEFRAME}.pkl'
MODEL_PATH = "trading_bot_hybrid"
LOOKBACK_WINDOW = 288

# Load data
df = pd.read_pickle(DATA_PATH)
print(f'✓ Loaded {len(df):,} rows for {DATA_SYMBOL} {DATA_TIMEFRAME}')

# Test on unseen data
total_timesteps = 1000
test_start = 5_536
test_data = df.iloc[test_start:test_start + total_timesteps].reset_index(drop=True)
print(f'✓ Testing on rows {test_start:,} to {test_start + total_timesteps:,}')

# Create test environment
test_env = SimpleTradingEnv(test_data, lookback_window=LOOKBACK_WINDOW)
test_env = Monitor(test_env)
test_env = DummyVecEnv([lambda: test_env])

# Load trained model
model = PPO.load(MODEL_PATH, env=test_env, device="cpu")
print(f'✓ Loaded model from {MODEL_PATH}\n')

# === FEATURE ACTIVATION TRACKING ===
extractor = model.policy.features_extractor
HOOKABLE_PATTERNS = ['_cnn', '_output', '_encoder', '_transformer', '_mlp', '_vp']

# Auto-discover all hookable modules
available_features = {}
for attr_name in dir(extractor):
    if attr_name.startswith('_'):
        continue
    if any(pattern in attr_name for pattern in HOOKABLE_PATTERNS):
        attr = getattr(extractor, attr_name)
        if isinstance(attr, torch.nn.Module):
            display_name = attr_name.replace('_', ' ').title().replace(' ', '_')
            available_features[attr_name] = display_name

✓ Loaded 264,323 rows for BTCUSDT 5m
✓ Testing on rows 5,536 to 6,536
Info: Dropped 99 rows due to NaNs after adding indicators.
✓ Loaded model from trading_bot_hybrid



In [2]:
from copy import deepcopy

# Setup hooks to capture activations
activations = {name: [] for name in available_features.keys()}

def make_hook(feature_name):
    def hook(module, input, output):
        # Capture mean absolute activation
        if isinstance(output, torch.Tensor):
            activations[feature_name].append(output.abs().mean().item())
    return hook

# Register hooks
hooks = []
for attr_name in available_features.keys():
    module = getattr(extractor, attr_name)
    hook = module.register_forward_hook(make_hook(attr_name))
    hooks.append(hook)

# Run evaluation
obs = test_env.reset()
done = False
total_reward = 0
episode_rewards = []
step_count = 0
last_env = None

while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = test_env.step(action)
    total_reward += reward[0]
    episode_rewards.append(reward[0])
    step_count += 1

    # Copy last environment state before fishing out or else will be resetted
    if test_env.envs[0].env.data_len == info[0].get('step') + 3:
        last_env = deepcopy(test_env.envs[0].env)

# Remove hooks
for hook in hooks:
    hook.remove()

# Print results
print("\n" + "="*80)
print("EVALUATION REPORT")
print("="*80)

print("\n📊 REWARD STATISTICS:")
print(f"   Total Reward:       {total_reward:+8.2f}")
print(f"   Min Reward:         {min(episode_rewards):+8.2f}")
print(f"   Max Reward:         {max(episode_rewards):+8.2f}")
print(f"   Avg Reward/Step:  {total_reward/step_count:+8.4f}")
print(f"   Total Steps:         {step_count:>6}")

# Feature activations
feature_stats = {name: np.mean(acts) if acts else 0.0 for name, acts in activations.items()}
sorted_features = sorted(feature_stats.items(), key=lambda x: x[1], reverse=True)

print(f"\n  Top 10 Most Active Features:")
for i, (name, avg_activation) in enumerate(sorted_features[:10], 1):
    display_name = available_features[name]
    print(f"    {i:2d}. {display_name:30s} : {avg_activation:.4f}")


EVALUATION REPORT

📊 REWARD STATISTICS:
   Total Reward:          +0.00
   Min Reward:            +0.00
   Max Reward:            +0.00
   Avg Reward/Step:   +0.0000
   Total Steps:            612

  Top 10 Most Active Features:
     1. Macro_Cnn                      : 0.7463
     2. Micro_Spatial_Cnn_Global       : 0.4428
     3. Vp_Bins_Cnn                    : 0.4265
     4. Micro_Spatial_Cnn_Medium       : 0.4207
     5. Vp_Levels_Continuous_Mlp       : 0.2464
     6. Account_Encoder                : 0.2385
     7. Meso_Cnn                       : 0.2062
     8. Vp_Levels_Binary_Mlp           : 0.1740
     9. Micro_Temporal_Cnn             : 0.1132
    10. Position_Encoder               : 0.0853


In [3]:
# === COMPREHENSIVE TRADING REPORT ===
print("\n" + "="*80)
print("TRADING PERFORMANCE REPORT")
print("="*80)

action_names = {0: 'HOLD', 1: 'LONG', 2: 'SHORT', 3: 'CLOSE'}

# Count actions from history
action_counts = {0: 0, 1: 0, 2: 0, 3: 0}
position_states = []

for env_state in last_env.history:
    action = env_state.get('action', [0])
    if isinstance(action, (list, np.ndarray)):
        action_id = int(action[0])
    else:
        action_id = int(action)
    action_counts[action_id] = action_counts.get(action_id, 0) + 1
    
    position_size = env_state.get('position_size', 0)
    position_states.append(position_size)

total_actions = sum(action_counts.values())

# === 1. ACTION DISTRIBUTION ===
print("\n📊 ACTION DISTRIBUTION:")
for action_id, count in sorted(action_counts.items()):
    pct = (count / total_actions * 100) if total_actions > 0 else 0
    bar = '█' * int(pct / 2)  # Visual bar
    print(f"   {action_names[action_id]:6s}: {count:5d} ({pct:5.1f}%) {bar}")

# === 2. POSITION DISTRIBUTION ===
flat_steps = sum(1 for ps in position_states if ps == 0)
long_steps = sum(1 for ps in position_states if ps > 0)
short_steps = sum(1 for ps in position_states if ps < 0)
total_steps = len(position_states)

print("\n📈 POSITION DISTRIBUTION:")
print(f"   FLAT : {flat_steps:5d} steps ({flat_steps/total_steps*100:5.1f}%)")
print(f"   LONG : {long_steps:5d} steps ({long_steps/total_steps*100:5.1f}%)")
print(f"   SHORT: {short_steps:5d} steps ({short_steps/total_steps*100:5.1f}%)")

# === 3. BALANCE PERFORMANCE ===
initial_balance = last_env.initial_balance
final_balance = last_env.broker.current_balance
balance_change = final_balance - initial_balance
balance_change_pct = (balance_change / initial_balance) * 100

print("\n💰 BALANCE PERFORMANCE:")
print(f"   Initial: ${initial_balance:>10,.2f}")
print(f"   Final:   ${final_balance:>10,.2f}")
print(f"   Change:  ${balance_change:>+10,.2f} ({balance_change_pct:+.2f}%)")

# === 4. COMPLETED TRADES ===
completed_trades = last_env.broker.trade_history
closed_trades = [t for t in completed_trades if t.get('status') == 'CLOSED']

if len(closed_trades) > 0:
    total_pnl = sum(t.get('pnl', 0) for t in closed_trades)
    wins = [t for t in closed_trades if t.get('pnl', 0) > 0]
    losses = [t for t in closed_trades if t.get('pnl', 0) <= 0]
    
    print(f"\n📋 TRADE SUMMARY:")
    print(f"   Total Trades:  {len(closed_trades):>5}")
    print(f"   Wins:          {len(wins):>5} ({len(wins)/len(closed_trades)*100:5.1f}%)")
    print(f"   Losses:        {len(losses):>5} ({len(losses)/len(closed_trades)*100:5.1f}%)")
    print(f"   Total PnL:     ${total_pnl:>+10,.2f}")
    print(f"   Avg PnL:       ${total_pnl/len(closed_trades):>+10,.2f}")
    if wins:
        print(f"   Avg Win:       ${sum(t['pnl'] for t in wins)/len(wins):>+10,.2f}")
    if losses:
        print(f"   Avg Loss:      ${sum(t['pnl'] for t in losses)/len(losses):>+10,.2f}")
    
    # Exit reason breakdown
    exit_reasons = {}
    for t in closed_trades:
        reason = t.get('reason', 'Unknown')
        exit_reasons[reason] = exit_reasons.get(reason, 0) + 1
    
    print(f"\n📊 EXIT REASONS:")
    for reason, count in sorted(exit_reasons.items(), key=lambda x: x[1], reverse=True):
        pct = (count / len(closed_trades)) * 100
        print(f"   {reason:20s}: {count:3d} ({pct:5.1f}%)")
    
    # === 5. DETAILED TRADE TABLE ===
    print("\n" + "="*80)
    print("DETAILED TRADE HISTORY")
    print("="*80)
    
    # Build step to action mapping
    step_to_action = {}
    for env_state in last_env.history:
        step = env_state.get('step', 0)
        action = env_state.get('action', [0])
        if isinstance(action, (list, np.ndarray)):
            action_id = int(action[0])
        else:
            action_id = int(action)
        step_to_action[step] = action_id
    
    # Create trades DataFrame
    trades_list = []
    for i, t in enumerate(closed_trades, 1):
        step_open = t.get('step_open', 0)
        action_id = step_to_action.get(step_open, 0)
        
        trades_list.append({
            '#': i,
            'Step': step_open,
            'Dir': action_names.get(action_id, '?'),
            'Entry': t.get('entry_price', 0),
            'Exit': t.get('exit_price', 0),
            'Duration': t.get('duration', 0),
            'PnL': t.get('pnl', 0),
            'PnL%': t.get('pnl_percent', 0) * 100,
            'Reason': t.get('reason', 'N/A'),
        })
    
    trades_df = pd.DataFrame(trades_list)
    
    # Apply color styling: green for profit, red for loss with black text
    def color_pnl(val):
        if val > 0:
            return 'background-color: #90EE90; color: black'  # Light green with black text
        elif val < 0:
            return 'background-color: #FF6B6B; color: black'  # Bright red with black text
        else:
            return 'color: black'
    
    # Display with styling (using map instead of deprecated applymap)
    styled_df = trades_df.style.format({
        'Entry': '${:,.2f}',
        'Exit': '${:,.2f}',
        'PnL': '${:+,.2f}',
        'PnL%': '{:+.2f}%',
        'Duration': '{:.0f}',
    }).map(color_pnl, subset=['PnL'])
    
    display(styled_df)

else:
    print(f"\n⚠️  NO COMPLETED TRADES")

# === 6. OPEN POSITION (if any) ===
final_state = last_env.history[-1]
final_position = final_state.get('position_size', 0)

if final_position != 0:
    direction = 'LONG' if final_position > 0 else 'SHORT'
    unrealized_pnl = final_state.get('unrealized_pnl', 0)
    entry_price = final_state.get('entry_price', 0)
    current_price = final_state.get('current_price', 0)
    sl_price = final_state.get('stop_loss_price', None)
    tp_price = final_state.get('take_profit_price', None)
    
    print(f"\n⚠️  OPEN POSITION:")
    print(f"   Direction:      {direction}")
    print(f"   Entry Price:    ${entry_price:,.2f}")
    print(f"   Current Price:  ${current_price:,.2f}")
    print(f"   Stop Loss:      ${sl_price:,.2f}" if sl_price is not None else "   Stop Loss:      None")
    print(f"   Take Profit:    ${tp_price:,.2f}" if tp_price is not None else "   Take Profit:    None")
    print(f"   Unrealized PnL: ${unrealized_pnl:+,.2f}")

print("\n" + "="*80)



TRADING PERFORMANCE REPORT

📊 ACTION DISTRIBUTION:
   HOLD  :   899 (100.0%) ██████████████████████████████████████████████████
   LONG  :     0 (  0.0%) 
   SHORT :     0 (  0.0%) 
   CLOSE :     0 (  0.0%) 

📈 POSITION DISTRIBUTION:
   FLAT :   899 steps (100.0%)
   LONG :     0 steps (  0.0%)
   SHORT:     0 steps (  0.0%)

💰 BALANCE PERFORMANCE:
   Initial: $ 10,000.00
   Final:   $ 10,000.00
   Change:  $     +0.00 (+0.00%)

⚠️  NO COMPLETED TRADES



## Trading Chart Visualization

Interactive candlestick chart with position markers and equity curve.

In [4]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def create_trading_chart(env_history, env_data):
    """
    Create candlestick chart with position markers.
    Uses env_data (not test_data) because env drops NaN rows during initialization.
    """
    
    # Detect position changes
    position_events = []
    for i in range(1, len(env_history)):
        prev_pos = env_history[i-1].get('position_size', 0)
        curr_pos = env_history[i].get('position_size', 0)
        step_idx = env_history[i].get('step')
        
        if prev_pos == 0 and curr_pos != 0:
            # Position opened
            position_events.append({
                'type': 'open',
                'step': step_idx,
                'price': env_history[i].get('entry_price', 0),
                'direction': 'LONG' if curr_pos > 0 else 'SHORT',
                'sl': env_history[i].get('stop_loss_price'),
                'tp': env_history[i].get('take_profit_price'),
            })
        elif prev_pos != 0 and curr_pos == 0:
            # Position closed
            position_events.append({
                'type': 'close',
                'step': step_idx,
                'price': env_history[i].get('current_price', 0),
                'direction': 'LONG' if prev_pos > 0 else 'SHORT',
                'pnl': env_history[i].get('realized_pnl', 0),
            })
    
    print(f"Found {len(position_events)} position events:")
    print(f"  Opens: {sum(1 for e in position_events if e['type'] == 'open')}")
    print(f"  Closes: {sum(1 for e in position_events if e['type'] == 'close')}")
    print(f"Environment data length: {len(env_data)}")
    
    # Create figure with 3 subplots
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        subplot_titles=('Price & Trading Activity', 'Account Equity', 'Rewards'),
        row_heights=[0.5, 0.25, 0.25]
    )
    
    # Add candlestick using env_data (which has correct indices)
    fig.add_trace(
        go.Candlestick(
            x=env_data.index,
            open=env_data['open'],
            high=env_data['high'],
            low=env_data['low'],
            close=env_data['close'],
            name='Price',
            increasing_line_color='#26a69a',
            decreasing_line_color='#ef5350'
        ),
        row=1, col=1
    )
    
    # Track open positions for drawing SL/TP lines
    open_positions = []
    
    # Add position markers
    for event in position_events:
        step = event['step']
        price = event['price']
        direction = event['direction']
        
        if event['type'] == 'open':
            # Entry marker
            color = '#00ff00' if direction == 'LONG' else '#ff0000'
            symbol = 'triangle-up' if direction == 'LONG' else 'triangle-down'
            
            fig.add_trace(
                go.Scatter(
                    x=[step],
                    y=[price],
                    mode='markers',
                    marker=dict(size=15, color=color, symbol=symbol, line=dict(width=2, color='white')),
                    name=f'{direction} Open',
                    showlegend=False,
                    hovertext=f'{direction} ENTRY<br>Step: {step}<br>Price: ${price:,.2f}',
                    hoverinfo='text'
                ),
                row=1, col=1
            )
            
            open_positions.append(event)
            
        else:  # close
            # Exit marker
            pnl = event.get('pnl', 0)
            color = '#90EE90' if pnl > 0 else '#FF6B6B'
            
            fig.add_trace(
                go.Scatter(
                    x=[step],
                    y=[price],
                    mode='markers',
                    marker=dict(size=12, color=color, symbol='x', line=dict(width=2, color='black')),
                    name=f'{direction} Close',
                    showlegend=False,
                    hovertext=f'{direction} EXIT<br>Step: {step}<br>Price: ${price:,.2f}<br>PnL: ${pnl:+,.2f}',
                    hoverinfo='text'
                ),
                row=1, col=1
            )
            
            # Find matching open and draw SL/TP lines
            if open_positions:
                for idx in range(len(open_positions) - 1, -1, -1):
                    if open_positions[idx]['direction'] == direction:
                        open_event = open_positions.pop(idx)
                        open_step = open_event['step']
                        
                        # Draw SL line
                        if open_event['sl'] is not None:
                            fig.add_shape(
                                type='line',
                                x0=open_step, x1=step,
                                y0=open_event['sl'], y1=open_event['sl'],
                                line=dict(color='red', width=1, dash='dash'),
                                row=1, col=1
                            )
                        
                        # Draw TP line
                        if open_event['tp'] is not None:
                            fig.add_shape(
                                type='line',
                                x0=open_step, x1=step,
                                y0=open_event['tp'], y1=open_event['tp'],
                                line=dict(color='green', width=1, dash='dash'),
                                row=1, col=1
                            )
                        break
    
    # Add equity curve
    steps = [s.get('step') for s in env_history]
    equity = [s.get('equity', 0) for s in env_history]
    
    fig.add_trace(
        go.Scatter(
            x=steps,
            y=equity,
            mode='lines',
            name='Equity',
            line=dict(color='#2196F3', width=2),
            fill='tozeroy',
            fillcolor='rgba(33, 150, 243, 0.1)'
        ),
        row=2, col=1
    )
    
    # Add rewards
    rewards = [s.get('reward', 0) for s in env_history]
    reward_colors = ['#90EE90' if r > 0 else '#FF6B6B' if r < 0 else '#888888' for r in rewards]
    
    fig.add_trace(
        go.Bar(
            x=steps,
            y=rewards,
            name='Reward',
            marker=dict(
                color=reward_colors,
                line=dict(width=0)
            ),
            hovertemplate='Step: %{x}<br>Reward: %{y:.4f}<extra></extra>'
        ),
        row=3, col=1
    )
    
    # Update layout
    fig.update_layout(
        title='Trading Activity Visualization',
        xaxis3_title='Step',
        yaxis_title='Price ($)',
        yaxis2_title='Equity ($)',
        yaxis3_title='Reward',
        height=1000,
        template='plotly_dark',
        hovermode='closest',
        showlegend=False
    )
    
    fig.update_xaxes(rangeslider_visible=False)
    
    return fig

# Generate chart using environment's data (not test_data)
print("\n" + "="*80)
print("Creating Trading Chart...")
print("="*80)
fig = create_trading_chart(last_env.history, last_env.data)
fig.show()
print("\n✓ Chart complete!")


Creating Trading Chart...
Found 0 position events:
  Opens: 0
  Closes: 0
Environment data length: 901



✓ Chart complete!
